In [3]:
import json

import polars as pl
from phoenix.client import client

from social_groups.reporting.group_reply import MajorityVote
from social_groups.reporting.parsing import (
    _ANSWER_OPTIONS,
    AnswerOptions,
    get_string_parser,
)
from social_groups.trialrunner.data_connectors.mmlu_pro import MMLUProExample
from social_groups.trialrunner.data_connectors.mmlu_pro_subset import (
    MMLUProSubsetConnector,
)
from social_groups.trialrunner.decision_schemes.base import ExampleOutput


In [13]:
projects: list[str] = [
    # "tribal_council - 2026-02-22-18-10-27",
    "tribal_council - 2026-02-23-19-02-29",
]

data = list(MMLUProSubsetConnector().iterate_data())


In [22]:
parser = get_string_parser(
    {
        r'(?i)(?:\b|\(|^|\s)([A-J])(?::?\)|\b|$|:)',
        r'(?<!\w)[A-J](?!\w)'
    },
    _ANSWER_OPTIONS[AnswerOptions.letters_A_to_J],
)

group_reply = MajorityVote()

In [14]:
def get_results_from_phoenix(experiment_name: str) -> list[tuple[ExampleOutput, MMLUProExample]]:
    d = client.Client().spans.get_spans_dataframe(project_identifier=experiment_name, limit=100000000, timeout=100)
    x = pl.DataFrame(d).sort("context.trace_id", "start_time", "end_time")

    x = x.filter(pl.col("attributes.output").is_not_null() | pl.col("attributes.question").is_not_null()).select(
        "attributes.output", "attributes.question")

    return [
        (ExampleOutput(**json.loads(i[0]), history=[]) if i[0] else None,
         MMLUProExample(**json.loads(i[1])) if i[1] else None) for i in
        x.iter_rows()
    ]

In [15]:
original_input: list[MMLUProExample] = sorted(data, key=lambda x: x.question)
answers = {project: sorted(get_results_from_phoenix(project), key=lambda x: x[1].question) for project in projects}

In [16]:
combined: dict[int, dict[str, ExampleOutput]] = {i.question_id: {} for i in original_input}

questions = {i.question_id: i for i in original_input}

for project, data in answers.items():
    for out, example in data:
        combined[example.question_id][project] = out


In [26]:
from social_groups.reporting.parsing import ParsingResultError

for project in projects:
    not_answered_questions = 0
    missing_responses = 0
    correct_answers = 0
    incorrect_answers = 0

    correct_in_proposed = 0
    incorrect_after_correct_in_proposed = 0

    answers = []
    for q_id, question in combined.items():
        if project not in question:
            not_answered_questions += 1
            continue
        if not question[project]:
            missing_responses += 1
            continue
        parsed_before = [parser(a) for a in question[project].answers_at_beginning]
        if questions[q_id].answer in parsed_before:
            correct_in_proposed += 1

        parsed = [parser(a) for a in question[project].answers_at_end]
        if any(p == ParsingResultError.NOT_PARSABLE.value for p in parsed):
            print(f"invalid: {question[project].answers_at_end}")
            print(parsed)
            print(" -- ")
        answer = group_reply(parsed)
        if answer == questions[q_id].answer:
            correct_answers += 1
        else:
            if questions[q_id].answer in parsed_before:
                incorrect_after_correct_in_proposed += 1
            incorrect_answers += 1
            # print(f"incorrect: soll: {questions[q_id].answer}, ist: {answer}")

    print(f"Project {project}")
    print(f"Total Questions: {len(combined)}")
    print(f"      - not answered: {not_answered_questions}")
    print(f"      - missing responses: {missing_responses}")
    print(f"Correct Answers: {correct_answers}")
    print(f"Incorrect Answers: {incorrect_answers}")
    print(f"Correct In Proposed: {correct_in_proposed}")
    print(f"Incorrect, but correct in proposed: {incorrect_after_correct_in_proposed}")
    print("------------------------------------------")

invalid: ['allergy to eggs', 'D', 'D']
['___not_parsable___', 'D', 'D']
 -- 
invalid: ['A', 'Clostridium difficile', 'A']
['A', '___not_parsable___', 'A']
 -- 
invalid: ['C', 'Proposal 0: 500 MHz = 16.850 mT = 0.01667 cm⁻¹', 'Proposal 0']
['C', '___not_parsable___', '___not_parsable___']
 -- 
invalid: ['163', 'H', 'H']
['___not_parsable___', 'H', 'H']
 -- 
invalid: ['Decreases, Decreases, Increases', 'Decreases, Decreases, Increases', 'Decreases, Decreases, Increases']
['___not_parsable___', '___not_parsable___', '___not_parsable___']
 -- 
invalid: ['C', 'C', "women's susceptibility to stress depended on their levels of social support"]
['C', 'C', '___not_parsable___']
 -- 
invalid: ['20.3%', '20.3%', '20.3%']
['___not_parsable___', '___not_parsable___', '___not_parsable___']
 -- 
invalid: ['7.4', 'E', 'E']
['___not_parsable___', 'E', 'E']
 -- 
invalid: ['distance decay', 'distance decay', 'distance decay']
['___not_parsable___', '___not_parsable___', '___not_parsable___']
 -- 
invalid